# 00 · Objetivo de analisis

> **Objetivo del cuaderno:** completar la **fase 1 de CRISP-DM** para el caso
> *Online Retail II*. No tocamos los datos aun: declaramos el **problema**,
> el **objetivo de analisis** y el **criterio de exito** para evitar que la
> ingenieria se adelante a las preguntas de negocio.

## CRISP-DM, fase 1 en una tabla

| Campo | Contenido |
|---|---|
| **Problema de negocio** | El retailer trata a todos los clientes por igual; las campañas de regalo rinden poco porque no hablan al cliente adecuado en el momento adecuado. |
| **Pregunta de negocio** | ¿Como agrupar a los clientes para que las acciones de marketing (comunicacion, descuentos, retencion) tengan rendimiento diferenciado? |
| **Objetivo de analisis** | Construir **segmentos no supervisados** de clientes basados en RFM extendido, estables entre K-Means y DBSCAN, y acompañarlos de una ficha accionable por segmento. |
| **Criterio de exito analitico** | Silhouette ≥ 0,35 **y** mas del 70 % de los clientes concentrados en 3-5 segmentos interpretables. |
| **Criterio de exito de negocio** | Que marketing adopte las fichas de accion generadas (medible en U4 por tasa de contacto por segmento). |
| **Riesgo principal** | Que el pipeline no sea reproducible el dia de grabacion. Mitigacion: `scripts/figuras_u2.py` y `tests/test_cifras_referencia.py`. |
| **Datos y licencia** | Online Retail II (UCI 502, Chen 2019, CC BY 4.0). Ver `data/README.md`. |
| **Alcance** | Clientes del retailer entre 2009-12-01 y 2011-12-09. Fuera: catalogo, trafico web, marketing historico. |

## Partes interesadas

- **Equipo de marketing:** consumidor de la ficha de accion.
- **Equipo de datos (nosotros):** responsable del pipeline y de la
  reproducibilidad.
- **Negocio (compras / almacen):** interesado en entender si hay productos
  "atadores" por segmento (U4).

## Entregables de la Unidad 2 (00-05)

1. Dataset validado + Parquet (`data/raw/online_retail_II.parquet`).
2. Reglas de limpieza en `segmentacion.data.limpieza` + pruebas rapidas.
3. Variables por cliente via `construir_rfm(ventas, fecha_corte)`.
4. **`HALLAZGOS.md`** con hallazgos, evidencia y decision para U3.
5. Perfil HTML (`reports/perfil_muestra.html`, gitignored) para difusion.

In [ ]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from segmentacion.config import PROCESSED_DATA_DIR, PROJECT_ROOT, RAW_DATA_DIR

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RAW_DATA_DIR    : {RAW_DATA_DIR}")
print(f"PROCESSED_DIR   : {PROCESSED_DATA_DIR}")
print("Checklist de ejecucion de la unidad:")
print(" 1) uv sync --extra segmentacion-clientes --extra dev")
print(" 2) uv run python scripts/descargar_datos.py")
print(" 3) ejecutar 01 -> 02 -> 04 -> 03 -> 05")
print(" 4) uv run pytest  (rapido) y uv run pytest -m slow (cifras)")

## Para discutir en clase

Preguntas de reflexión derivadas de las **fuentes primarias de CRISP-DM**
(Chapman et al., 2000) y del libro *Data Mining for Business Analytics*
(Shmueli et al.). Todas se aplican al caso de Online Retail II.

1. **Fase 1 (Business Understanding).** CRISP-DM pide escribir el
   problema de negocio ANTES de ver los datos. ¿Qué cambiaría si
   descubrimos a mitad de la Unidad 2 que la mayoría de clientes son
   mayoristas (comportamiento muy distinto al B2C que asumimos)?
   ¿Volvemos a la fase 1, o seguimos con la segmentación B2C?

2. **Entre fases.** CRISP-DM es cíclica: si en la Unidad 3 el modelo
   no supera a la regla de recencia por un margen razonable, ¿qué
   fase conviene revisitar primero? ¿La definición del problema (U2),
   la limpieza (U2), la ingeniería de variables (U3) o los modelos (U3)?

3. **Criterio de éxito.** El Silhouette ≥ 0,35 que fijamos es un
   criterio **analítico**. ¿Cómo lo traducirías a un criterio de
   **negocio** verificable en U4? Pista: cuántos clientes activa
   marketing por mes, por segmento.

4. **Riesgos.** CRISP-DM pide listar los riesgos del proyecto. ¿Cuál
   crees que es el riesgo más probable aquí: (a) que las cifras del
   OVA no se reproduzcan el día de la grabación, (b) que marketing
   no adopte las fichas de acción, o (c) que un cambio en el formato
   del archivo UCI rompa el flujo?

5. **Fuentes de datos.** El dataset viene de un único retailer
   británico (2009-2011). ¿Qué sesgos introduce esto para generalizar
   los segmentos a otros retailers? ¿Y a otros años (recesión 2008,
   pandemia 2020)?

6. **Ética.** Si marketing decide **excluir** a los clientes
   etiquetados como "Inactivos / Perdidos" de todas las campañas,
   ¿qué efecto tiene sobre la equidad? ¿Son los segmentos
   suficientemente objetivos para una decisión así, o deberíamos
   publicar una tasa de contacto mínima por segmento?

## Decisiones ya tomadas (son contrato)

- **Formato de trabajo:** Parquet, no CSV (`data/raw/online_retail_II.parquet`).
- **Esquema explicito al leer:** `Invoice` y `StockCode` como `Utf8`,
  `Customer ID` como `Int64`. Vive en `segmentacion.data.cargar.ESQUEMA`.
- **Separacion por responsabilidad:**
  - U2 queda con 00-05 (comprender, limpiar, construir y explorar).
  - U3 queda con 06-10 (particion, feature engineering, pipelines,
    clustering).
- **Reproducibilidad numerica:** las cifras del OVA (seccion 4 de la
  spec) se verifican en `tests/test_cifras_referencia.py` marcado como
  `slow`. Si cambian, el test falla y hay que actualizar el OVA y la
  spec a la vez — no la prueba.

## Siguiente paso

- `01_comprension_datos.ipynb` — lectura con esquema y diagnostico.